# Full-scale vanilla SegFormer-B0 training + Figure 2 regeneration

One-zip version of Kalana-Person2's full-scale training pipeline. Trains **only** the vanilla baseline (~2h GPU) — the λ2=1.0 attention checkpoint is already bundled in this zip and gets auto-seeded into the output folder below, so it's skipped rather than retrained. Then evaluates both checkpoints and regenerates the qualitative attention-drift figures (Figure 2) with the real checkpoints.

Upload **only** `vanilla_train_colab.zip` (from `make_vanilla_train_colab_zip.py`), not the whole repo.

**Before running:** Runtime → Change runtime type → GPU (T4 or better). Upload the zip to `MyDrive/`.

## Step 0: Unzip + auto-seed the attention checkpoint

Checkpoints/results write to `MyDrive/segformer_full_scale_outputs` so a dropped runtime does not lose the run. The bundled `segformer_b0_att_best.pt` gets copied there now, once, so Step 3's per-variant skip-if-exists check skips attention training.

In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/vanilla_train_colab.zip")
BUNDLE = Path("/content/segformer_full_scale")
OUTPUTS = Path("/content/drive/MyDrive/segformer_full_scale_outputs")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    OUTPUTS = HERE

sys.path.insert(0, str(HERE))

# Auto-seed: copy any bundled checkpoints into the persistent output dir
# (once) so Step 3's skip-if-exists guard leaves them untouched.
seed_ckpt_dir = HERE / "checkpoints"
out_ckpt_dir = OUTPUTS / "checkpoints"
out_ckpt_dir.mkdir(parents=True, exist_ok=True)
for src in seed_ckpt_dir.glob("*.pt"):
    dst = out_ckpt_dir / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
        print("Seeded", dst)
    else:
        print("Already present, not overwriting:", dst)

print("HERE =", HERE)
print("OUTPUTS =", OUTPUTS)

In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")

In [ ]:
import paths
from paths import (
    DATA_IMG_DIR, DATA_MASK_DIR, PERSON3_DIR, PERSON4_DIR, LAYOUT_MODE,
    N_TRAIN, N_VAL, N_TEST, SEED, EPOCHS,
    add_teammate_paths, apply_data_dirs, set_output_dirs,
)

set_output_dirs(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()

print("layout:          ", LAYOUT_MODE)
print("Person 3 package:", PERSON3_DIR, "->", (PERSON3_DIR / "attention_consistency").is_dir())
print("images:", DATA_IMG_DIR.is_dir(), DATA_IMG_DIR)
print("masks:", DATA_MASK_DIR.is_dir(), DATA_MASK_DIR)
print("att checkpoint pre-seeded:", (paths.CKPT_DIR / "segformer_b0_att_best.pt").exists())
assert (PERSON3_DIR / "attention_consistency").is_dir()
assert DATA_IMG_DIR.is_dir() and DATA_MASK_DIR.is_dir()

## Step 1: Device check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Full-scale 20-epoch training will not finish in a reasonable time.")

## Step 2: Split preflight

Confirms the 3576/766/766 seed-42 lists match Chanupa's U-Net split algorithm before any GPU time is spent.

In [ ]:
import runpy
runpy.run_path(str(HERE / "tests" / "test_split_identity.py"), run_name="__main__")

## Step 3: Train (vanilla only — attention is pre-seeded and gets skipped)

In [ ]:
import train_full_scale as T
import paths
from paths import BATCH_SIZE, LR, LAMBDA2, SIGMA, ATT_MODE, SEED, EPOCHS, N_TRAIN, N_VAL, N_TEST

class Args:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    epochs = EPOCHS
    batch_size = BATCH_SIZE
    lr = LR
    lambda2 = LAMBDA2
    sigma = SIGMA
    att_mode = ATT_MODE
    seed = SEED

args = Args()
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

for variant in ("vanilla", "att"):
    best = paths.CKPT_DIR / f"segformer_b0_{variant}_best.pt"
    if best.exists():
        print(f"SKIP train {variant}: {best.name} already exists")
        continue
    T.train_variant(variant, args)

## Step 4: Evaluate both checkpoints

In [ ]:
import eval_full_scale as E

class EvalArgs:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    seed = SEED

rows = []
for variant in ("vanilla", "att"):
    rows.append(E.evaluate_variant(variant, EvalArgs()))
print("\nFull-scale rows:")
for r in rows:
    print(f"  {r['model']}: dice={r['dice']} iou={r['iou']} aamo={r['aamo']}")

## Step 5: Qualitative attention-drift figures (Figure 2)

In [ ]:
import generate_full_scale_figures as G
import sys
sys.argv = ["generate_full_scale_figures.py", "--n", "3"]
G.main()

## Step 6: Download results

In [ ]:
import paths

print("outputs under", paths.RESULTS_DIR)
for p in sorted(paths.CKPT_DIR.glob("*.pt")):
    print(" ckpt", p.name, p.stat().st_size)
for p in sorted(paths.RESULTS_DIR.rglob("*")):
    if p.is_file():
        print(" result", p.relative_to(paths.RESULTS_DIR))

if IN_COLAB:
    from google.colab import files
    vanilla_ckpt = paths.CKPT_DIR / "segformer_b0_vanilla_best.pt"
    if vanilla_ckpt.exists():
        files.download(str(vanilla_ckpt))
    fig_dir = paths.RESULTS_DIR / "attention_drift_figures"
    for p in sorted(fig_dir.glob("attention_drift_*_full_scale.png")):
        files.download(str(p))